# LFW 03. Compressed materialization and index

## 예상 소요 시간 (현재 LFW 13,195개 임베딩·로컬 PostgreSQL 기준)

| 실행 모드 | 예상 시간 | 주로 오래 걸리는 구간 |
| --- | ---: | --- |
| `EXECUTE_STAGE=False` | 1초 미만 | run 연결과 이전 phase 확인 준비 |
| `EXECUTE_STAGE=True` | 약 5~30분 | 전체 변환, PCA/PQ 약 26,000건 저장, HNSW index |

> 30초 heartbeat와 DB 저장 500건 단위 진행률을 출력합니다. DB 상태와 기존 저장 건수에 따라 시간 차이가 큽니다.

목표: 02에서 고정한 compressor로 전체 원본 임베딩을 변환하고, PCA 검색 벡터와 PQ code를 별도 DB 테이블에 저장한 뒤 pgvector index를 확인합니다. 성공 기준은 source/model attempt가 각 행과 phase log에 연결되는 것입니다.

> **재시작/재개 규칙(필수)**: 임의 셀에서 시작하지 말고 **Kernel Restart 후 Run All**을 사용합니다. 02의 가장 최근 `completed` attempt만 사용합니다. 중단되면 03 전체를 다시 실행하며 같은 run/model의 기존 행은 건너뜁니다. compressor나 원본 임베딩이 바뀌면 02 또는 01부터 다시 시작하고, protocol/config가 바뀌면 00부터 새 run을 만듭니다.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate.resolve()
    raise RuntimeError('Run Jupyter from the ronbun repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
from research.runtime import ProgressReporter, RunStore, resolve_active_run

EXECUTE_STAGE = False
RUN_ROOT = PROJECT_ROOT / 'runs' / 'lfw'
LEGACY_RUN_ROOT = PROJECT_ROOT / 'runs'
try:
    RUN_DIR = resolve_active_run(RUN_ROOT)
except FileNotFoundError:
    RUN_ROOT = LEGACY_RUN_ROOT
    RUN_DIR = resolve_active_run(RUN_ROOT)
PROGRESS = ProgressReporter('03 materialization/index', heartbeat_seconds=30)


## Plan

- Resolve one completed compressor attempt and load its immutable artifacts.
- Materialize PCA-256 as a searchable pgvector and PQ as non-searchable binary codes.
- Record insert/skip counts plus reconstruction/angular error summaries; create vector indexes.


In [ ]:
def attach_run(run_dir: Path) -> tuple[RunStore, dict]:
    run = RunStore.open(run_dir)
    manifest = json.loads(run.manifest_path.read_text(encoding='utf-8'))
    if manifest.get('status') == 'completed' or (run_dir / 'COMPLETED').exists():
        raise RuntimeError('Completed runs are immutable.')
    return run, manifest

def latest_completed_attempt(run_dir: Path, phase_name: str) -> int:
    attempts = run_dir / 'phases' / phase_name / 'attempts'
    completed = []
    for path in sorted(attempts.glob('A*/phase_manifest.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        if payload.get('status') == 'completed':
            completed.append(int(payload['attempt']))
    if not completed:
        raise RuntimeError(f'No completed attempt for {phase_name}.')
    return max(completed)

preflight = {'execute_stage': EXECUTE_STAGE, 'run_dir_resolved': str(RUN_DIR),
             'run_manifest_exists': bool(RUN_DIR and (RUN_DIR / 'run_manifest.json').is_file())}
preflight


## Execute and record

PCA retrieval vector와 512D reconstructed certificate vector를 혼동하지 마십시오. PQ code는 pgvector index 대상이 아닙니다.


In [ ]:
result = {'status': 'not_executed', **preflight}
if EXECUTE_STAGE:
    import pandas as pd
    from research.compression import PCACompressor, PQCompressor
    from research.database import create_database_engine, load_database_settings
    from research.experiments import materialize_compressed_embeddings
    from research.runtime.hashing import sha256_file

    with PROGRESS.step('run/input 및 02 artifact 검증', expected='10초 미만'):
        run, run_manifest = attach_run(RUN_DIR)
        run.verify_inputs()
        run.verify_phase_artifacts('02_compressor_fit')
        compressor_attempt = latest_completed_attempt(RUN_DIR, '02_compressor_fit')
        compressor_suffix = f'A{compressor_attempt:03d}'
        artifact_dir = RUN_DIR / 'artifacts' / '02_compressor_fit'
        pca_path = artifact_dir / f'pca_256_{compressor_suffix}.joblib'
        pq_path = artifact_dir / f'pq_{compressor_suffix}.faiss'
        if not pca_path.is_file() or not pq_path.is_file():
            raise FileNotFoundError({'pca': str(pca_path), 'pq': str(pq_path)})
        pca = PCACompressor.load(pca_path)
        pq = PQCompressor.load(pq_path)
        config = run_manifest['config']
        manifest = pd.read_csv(PROJECT_ROOT / config['dataset']['manifest_path'])
        development_paths = {
            str((PROJECT_ROOT / Path(str(path))).resolve())
            if not Path(str(path)).is_absolute()
            else str(Path(str(path)).resolve())
            for path in manifest.loc[manifest['split'].eq('development'), 'image_path']
        }
        if not development_paths:
            raise ValueError('development image paths are required for frozen normalization.')
        engine = create_database_engine(load_database_settings())

    with run.phase('03_compressed_materialization_and_index') as phase:
        suffix = f'A{phase.attempt:03d}'
        measurements_path = phase.attempt_dir / f'compression_measurements_{suffix}.csv'

        def report(message: str, details: dict[str, object]) -> None:
            PROGRESS.emit(message, **details)

        with PROGRESS.step(
            'development 오차 통계 고정 및 전체 배치 materialization',
            expected='10~60분(데이터·DB 상태에 따라 증가)',
        ):
            summary = materialize_compressed_embeddings(
                engine,
                run_uid=run.run_id,
                pca=pca,
                pq=pq,
                pca_artifact_path=pca_path,
                pca_artifact_sha256=sha256_file(pca_path),
                pq_artifact_path=pq_path,
                pq_artifact_sha256=sha256_file(pq_path),
                development_image_paths=development_paths,
                measurements_path=measurements_path,
                batch_size=512,
                progress=report,
            )
        summary['compressor_attempt'] = compressor_suffix
        summary_path = phase.attempt_dir / f'materialization_summary_{suffix}.json'
        summary_path.write_text(
            json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        phase.publish_artifact(measurements_path)
        phase.publish_artifact(summary_path)
        phase.record_counts(**summary['counts'])
    PROGRESS.emit('03 완료', **summary['counts'])
    result = {'status': 'completed', 'run_id': run.run_id, **summary}
else:
    PROGRESS.emit('검토 모드 완료: 변환/DB 저장/index 생성을 실행하지 않음', expected='1초 미만')
result


## Next step

insert/skip 합계가 source vector 수와 맞는지 확인합니다. 불일치나 index 실패가 있으면 03부터 재시작하고, 변환 모델 변경은 02부터 새 attempt로 수행합니다.
